In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/dlp-nppe-1-t-22026/sample_submission.csv
/kaggle/input/competitions/dlp-nppe-1-t-22026/train.csv
/kaggle/input/competitions/dlp-nppe-1-t-22026/test.csv


In [2]:
import os, re, gc
import pandas as pd
import numpy as np
import torch
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, DataCollatorWithPadding
)
from peft import LoraConfig, get_peft_model, TaskType

print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

CUDA available: True
Device: Tesla T4


In [3]:
train = pd.read_csv('/kaggle/input/competitions/dlp-nppe-1-t-22026/train.csv')
test = pd.read_csv('/kaggle/input/competitions/dlp-nppe-1-t-22026/test.csv')
sample_sub = pd.read_csv('/kaggle/input/competitions/dlp-nppe-1-t-22026/sample_submission.csv')

TEXT_COL = "text"      # confirm this matches your real column name
LABEL_COL = "label"

print(train.shape, test.shape)
print(train.columns.tolist())

(50840, 3) (12710, 2)
['id', 'text', 'label']


In [4]:
def clean_legal_text(text):
    text = str(text)
    text = re.sub(r'\s+', ' ', text)                        # collapse whitespace/newlines
    text = re.sub(r'Page \d+ of \d+', '', text, flags=re.I) # page numbers
    text = re.sub(r'-{3,}', '', text)                        # long dash separators
    text = re.sub(r'_{3,}', '', text)
    text = re.sub(r'\[.*?\]', '', text) # long underscore separators
    return text.strip()

def head_tail_truncate(text, tokenizer, max_len, head_ratio=0.75):
    tokens = tokenizer.encode(str(text), add_special_tokens=False)
    if len(tokens) <= max_len - 2:
        return text
    head_len = int((max_len - 2) * head_ratio)
    tail_len = (max_len - 2) - head_len
    new_tokens = tokens[:head_len] + tokens[-tail_len:]
    return tokenizer.decode(new_tokens)

# apply cleaning once, to all three sets (truncation happens per-model later, since it depends on the tokenizer)
train[TEXT_COL] = train[TEXT_COL].apply(clean_legal_text)
test[TEXT_COL] = test[TEXT_COL].apply(clean_legal_text)

In [5]:
label_list = sorted(train[LABEL_COL].unique())
label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for l, i in label2id.items()}
train['label_id'] = train[LABEL_COL].map(label2id)
NUM_LABELS = len(label_list)

train_df, val_df = train_test_split(
    train, test_size=0.2, random_state=42, stratify=train['label_id']
)

print("Classes:", NUM_LABELS)
print(train_df['label_id'].value_counts(normalize=True).sort_index())

Classes: 30
label_id
0     0.055001
1     0.104765
2     0.040642
3     0.050207
4     0.038872
5     0.069138
6     0.048338
7     0.138818
8     0.116788
9     0.045535
10    0.015293
11    0.065106
12    0.058886
13    0.027070
14    0.010990
15    0.020776
16    0.013080
17    0.003688
18    0.003737
19    0.036635
20    0.004745
21    0.001475
22    0.001918
23    0.003983
24    0.000688
25    0.013351
26    0.007671
27    0.000639
28    0.002016
29    0.000148
Name: proportion, dtype: float64


In [6]:
def get_target_modules(model_name):
    name = model_name.lower()
    if "distilbert" in name:
        return ["q_lin", "v_lin"]
    elif "deberta" in name:
        return ["query_proj", "value_proj"]
    elif "longformer" in name:
        return ["query", "value"]
    else:
        return ["query", "value"]  # bert, roberta, albert

def build_lora_model(model_name, num_labels):
    base_model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)
    lora_config = LoraConfig(
        task_type=TaskType.SEQ_CLS,
        r=8,
        lora_alpha=16,
        lora_dropout=0.1,
        target_modules=get_target_modules(model_name),
        modules_to_save=["classifier"],   # <-- critical: keeps classification head trainable
        bias="none",
    )
    model = get_peft_model(base_model, lora_config)
    return model

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)
    return {"accuracy": accuracy_score(labels, preds)}

In [7]:
def run_experiment(model_name, max_len, epochs=3, lr=2e-4, batch_size=8):
    print(f"\n{'='*60}\nRunning: {model_name} | max_len={max_len}\n{'='*60}")

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    # truncate per-model, since token counts differ by tokenizer
    tr = train_df.copy()
    va = val_df.copy()
    tr[TEXT_COL] = tr[TEXT_COL].apply(lambda x: head_tail_truncate(x, tokenizer, max_len))
    va[TEXT_COL] = va[TEXT_COL].apply(lambda x: head_tail_truncate(x, tokenizer, max_len))

    def tokenize(batch):
        return tokenizer(batch[TEXT_COL], truncation=True, max_length=max_len)

    train_ds = Dataset.from_pandas(tr[[TEXT_COL, 'label_id']].rename(columns={'label_id': 'labels'}))
    val_ds = Dataset.from_pandas(va[[TEXT_COL, 'label_id']].rename(columns={'label_id': 'labels'}))
    train_ds = train_ds.map(tokenize, batched=True, remove_columns=[TEXT_COL])
    val_ds = val_ds.map(tokenize, batched=True, remove_columns=[TEXT_COL])

    collator = DataCollatorWithPadding(tokenizer=tokenizer)
    model = build_lora_model(model_name, NUM_LABELS)
    model.print_trainable_parameters()

    training_args = TrainingArguments(
        output_dir=f"./results_{model_name.replace('/', '_')}",
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size * 2,
        learning_rate=lr,
        warmup_ratio=0.06,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="no",
        logging_steps=100,
        report_to="none",
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        processing_class=tokenizer,
        data_collator=collator,
        compute_metrics=compute_metrics,
    )

    trainer.train()
    metrics = trainer.evaluate()

    # free GPU memory before next model
    del model, trainer
    gc.collect()
    torch.cuda.empty_cache()

    return metrics["eval_accuracy"], tokenizer, training_args, max_len

In [8]:
!pip install -q -U torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 10.2 MB/s eta 0:00:00


In [9]:
models_to_try = [
    {"name": "distilbert-base-uncased", "max_len": 512},
    {"name": "bert-base-uncased", "max_len": 512},
    {"name": "microsoft/deberta-v3-base", "max_len": 512},
    # add "allenai/longformer-base-4096" with max_len=1024 later if you have GPU budget left
]

results = []
for cfg in models_to_try:
    acc, tok, targs, mlen = run_experiment(cfg["name"], cfg["max_len"], epochs=2)
    results.append({"model": cfg["name"], "val_accuracy": acc})

results_df = pd.DataFrame(results).sort_values("val_accuracy", ascending=False)
print(results_df)


Running: distilbert-base-uncased | max_len=512


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/371M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy
1,16.422283,14.452870,0.215873
2,7.679949,6.201665,0.286192


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


                       model  val_accuracy
0    distilbert-base-uncased      0.680665
1          bert-base-uncased      0.628737
2  microsoft/deberta-v3-base      0.286192


In [10]:
BEST_MODEL = results_df.iloc[0]["model"]   # or hardcode your choice
BEST_MAX_LEN = 512                          # match whatever you used above

# retrain with more epochs for the final run
final_tokenizer = AutoTokenizer.from_pretrained(BEST_MODEL)

tr = train_df.copy()
va = val_df.copy()
tr[TEXT_COL] = tr[TEXT_COL].apply(lambda x: head_tail_truncate(x, final_tokenizer, BEST_MAX_LEN))
va[TEXT_COL] = va[TEXT_COL].apply(lambda x: head_tail_truncate(x, final_tokenizer, BEST_MAX_LEN))

def tokenize(batch):
    return final_tokenizer(batch[TEXT_COL], truncation=True, max_length=BEST_MAX_LEN)

train_ds = Dataset.from_pandas(tr[[TEXT_COL, 'label_id']].rename(columns={'label_id': 'labels'}))
val_ds = Dataset.from_pandas(va[[TEXT_COL, 'label_id']].rename(columns={'label_id': 'labels'}))
train_ds = train_ds.map(tokenize, batched=True, remove_columns=[TEXT_COL])
val_ds = val_ds.map(tokenize, batched=True, remove_columns=[TEXT_COL])

collator = DataCollatorWithPadding(tokenizer=final_tokenizer)
model = build_lora_model(BEST_MODEL, NUM_LABELS)

training_args = TrainingArguments(
    output_dir="./final_model",
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    learning_rate=2e-4,
    warmup_ratio=0.06,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    logging_steps=50,
    report_to="none",
)

trainer = Trainer(
    model=model, args=training_args,
    train_dataset=train_ds, eval_dataset=val_ds,
    processing_class=final_tokenizer, data_collator=collator,
    compute_metrics=compute_metrics,
)
trainer.train()

Token indices sequence length is longer than the specified maximum sequence length for this model (1642 > 512). Running this sequence through the model will result in indexing errors


Map:   0%|          | 0/40672 [00:00<?, ? examples/s]

Map:   0%|          | 0/10168 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were

Epoch,Training Loss,Validation Loss,Accuracy
1,2.341598,2.237572,0.657750
2,2.051618,2.054234,0.681452
3,1.839030,1.955157,0.691581
4,1.675091,1.897156,0.699744
5,1.597313,1.877663,0.704563


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector

TrainOutput(global_step=12710, training_loss=2.079144047602812, metrics={'train_runtime': 5708.1841, 'train_samples_per_second': 35.626, 'train_steps_per_second': 2.227, 'total_flos': 2.742750880137216e+16, 'train_loss': 2.079144047602812, 'epoch': 5.0})

In [11]:
test_copy = test.copy()
test_copy[TEXT_COL] = test_copy[TEXT_COL].apply(lambda x: head_tail_truncate(x, final_tokenizer, BEST_MAX_LEN))
test_ds = Dataset.from_pandas(test_copy[[TEXT_COL]])
test_ds = test_ds.map(tokenize, batched=True, remove_columns=[TEXT_COL])

preds = trainer.predict(test_ds)
pred_ids = preds.predictions.argmax(axis=-1)
pred_labels = [id2label[i] for i in pred_ids]

submission = pd.DataFrame({
    'ID': test['id'].values,
    'label': pred_labels
})
submission.to_csv('submission.csv', index=False)
print(submission.shape)
submission.head()

Map:   0%|          | 0/12710 [00:00<?, ? examples/s]

(12710, 2)


,ID,label
0,0,7
1,1,2
2,2,3
3,3,11
4,4,0
